# Load And Plot All Predictions

This notebook reuses the current AICME loading path from `load_and_compute.ipynb` and exports one prediction plot per empirical Study/Drug row.

Workflow:
1. Load one experiment folder plus the selected checkpoint.
2. Read the empirical held-out batches already cached by the datamodule.
3. Use only `BATCH_INDEX = 0` for each dataset.
4. Slice every Study/Drug row inside that first batch.
5. Sample one prediction trajectory set per row and save one plot per drug.


In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch


def find_project_root(start: Path) -> Path:
    """Walk upward until the repository root is found."""

    for candidate in [start, *start.parents]:
        if (candidate / "pff").exists() and (candidate / "config_files").exists():
            return candidate
    raise RuntimeError("Could not find the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pff.config_classes.node_pk_config import NodePKExperimentConfig
from pff.data.data_empirical.builder import prediction_to_study_jsons
from pff.data.datasets.aicme_datasets import AICMECompartmentsDataModule
from pff.models.amortized_inference.aicme import AICMEPK
from pff.training.utils import (
    get_lightning_checkpoint_path,
    load_model_from_checkpoint_path,
)
from pff.utils.plots.databatch_plot import plot_study_json_with_prediction

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


def discover_experiment_dirs(results_dir: Path) -> list[Path]:
    """Return experiment folders ordered by modification time."""

    if not results_dir.exists():
        return []
    candidates = [
        path
        for path in results_dir.iterdir()
        if path.is_dir() and (path / "experiment_config.yaml").exists()
    ]
    return sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)


def load_aicme_experiment(
    experiment_dir: Path,
    *,
    checkpoint_type: str = "best",
    map_location: str = "cpu",
) -> tuple[NodePKExperimentConfig, AICMEPK, AICMECompartmentsDataModule, Path]:
    """Load one AICME model together with its empirical datamodule."""

    config_path = experiment_dir / "experiment_config.yaml"
    if not config_path.exists():
        raise FileNotFoundError(f"Missing experiment config: {config_path}")

    exp_config = NodePKExperimentConfig.from_yaml(str(config_path))
    checkpoint_path = get_lightning_checkpoint_path(str(experiment_dir), checkpoint_type)
    if checkpoint_path is None:
        raise FileNotFoundError(
            f"Could not find a '{checkpoint_type}' checkpoint inside {experiment_dir}."
        )

    model = load_model_from_checkpoint_path(
        AICMEPK,
        checkpoint_path,
        model_config=exp_config,
        map_location=map_location,
        strict=True,
    )
    model = model.to(map_location)
    model.eval()

    datamodule = AICMECompartmentsDataModule(exp_config)
    datamodule.setup()
    return exp_config, model, datamodule, Path(checkpoint_path)


def safe_name(name: object) -> str:
    """Convert labels into filesystem-safe path components."""

    text = str(name).strip().replace("/", "_")
    text = re.sub(r"[^0-9A-Za-z._-]+", "-", text)
    text = re.sub(r"-+", "-", text).strip("-._")
    return text or "unknown"


def split_dataset_key(dataset_key: str) -> tuple[str, str]:
    """Split `user/dataset` keys while tolerating unexpected formats."""

    parts = str(dataset_key).split("/", maxsplit=1)
    if len(parts) == 2:
        return parts[0], parts[1]
    return "unknown_user", parts[0]


def iter_first_batch_selections(
    datamodule: AICMECompartmentsDataModule,
    *,
    no_heldout: bool,
    batch_index: int,
):
    """Yield one selection per Study/Drug row from one fixed batch index."""

    batch_map = datamodule.get_empirical_test_batches(no_heldout=no_heldout)
    for dataset_key, batch_list in batch_map.items():
        if not batch_list:
            continue
        if batch_index >= len(batch_list):
            print(
                f"Skipping dataset '{dataset_key}': batch_index={batch_index} is out of range "
                f"for {len(batch_list)} available batch(es)."
            )
            continue

        raw_batch = batch_list[batch_index]
        studies, drugs = datamodule.describe_empirical_batch(raw_batch, print_available=False)
        for selection_index, drug_name in enumerate(drugs):
            study_name = (
                studies[selection_index]
                if selection_index < len(studies)
                else f"study_{selection_index:03d}"
            )
            yield {
                "dataset_key": dataset_key,
                "batch_index": batch_index,
                "selection_index": selection_index,
                "study_name": study_name,
                "drug_name": drug_name,
                "batch": datamodule.slice_single_substance_batch(raw_batch, selection_index),
            }


def selection_output_dir(selection: dict[str, object]) -> Path:
    """Build one output folder per dataset."""

    user_name, dataset_name = split_dataset_key(str(selection["dataset_key"]))
    return OUTPUT_ROOT / safe_name(EXPERIMENT_DIR.name) / safe_name(user_name) / safe_name(dataset_name)


def selection_file_stem(selection: dict[str, object]) -> str:
    """Build a unique filename stem for one Study/Drug selection."""

    return (
        f"batch_{int(selection['batch_index']):03d}"
        f"__row_{int(selection['selection_index']):03d}"
        f"__{safe_name(selection['study_name'])}"
        f"__{safe_name(selection['drug_name'])}"
    )


def save_prediction_plot(
    *,
    model: AICMEPK,
    batch,
    image_path: Path,
    device: torch.device,
    sample_size: int,
    log_scale: bool,
    plot_kwargs: dict[str, object],
) -> dict[str, str]:
    """Save one prediction plot and return study/drug metadata."""

    batch_device = batch.to(device)
    batch_cpu = batch.to("cpu")

    with torch.inference_mode():
        (
            prediction_samples,
            prediction_times,
            _target_future,
            _target_future_mask,
        ) = model.sample_individual_prediction(
            batch_device,
            sample_size=sample_size,
        )

    # prediction_samples : [S, B, It, Tr, 1]
    # prediction_times   : [S, B, It, Tr, 1]
    studies_with_predictions = prediction_to_study_jsons(
        prediction_samples.detach().cpu(),
        prediction_times.detach().cpu(),
        batch_cpu,
        model.meta_dosing,
    )
    study = studies_with_predictions[0]

    render_kwargs = dict(plot_kwargs)
    figure_size = tuple(render_kwargs.pop("figure_size", (8, 6)))
    title = render_kwargs.pop("title", None)
    title_font_size = render_kwargs.pop("title_font_size", None)

    fig, ax = plt.subplots(figsize=figure_size)
    plot_study_json_with_prediction(
        study,
        ax=ax,
        log_scale=log_scale,
        **render_kwargs,
    )
    resolved_title = title or (
        f"{study['meta_data']['study_name']} | {study['meta_data']['substance_name']}"
    )
    if title_font_size is not None:
        ax.set_title(resolved_title, fontsize=float(title_font_size))
    else:
        ax.set_title(resolved_title)

    image_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(image_path, bbox_inches="tight")
    plt.close(fig)

    return {
        "study_name": str(study["meta_data"]["study_name"]),
        "drug_name": str(study["meta_data"]["substance_name"]),
    }


In [2]:
RESULTS_DIR = PROJECT_ROOT / "results" / "comet" / "uai"
AVAILABLE_EXPERIMENT_DIRS = discover_experiment_dirs(RESULTS_DIR)

# Change these values for the experiment you want to inspect.
EXPERIMENT_DIR = Path("/home/cesarali/Pharma/pff/results/comet/uai/cb906bbb30f34522813c1357fff45371")
CHECKPOINT_TYPE = "log_rmse"  # "best", "last", or scheduler metric names such as "log_rmse"
MAP_LOCATION = "cpu"  # "cpu" or "cuda"
NO_HELDOUT = False
BATCH_INDEX = 0
FIX_PAST_TARGET_STEPS = 5
SAMPLE_SIZE = 1
PLOT_LOG_SCALE = True
OUTPUT_ROOT = PROJECT_ROOT / "reports" / "results_aistats" / "all_predictions"

predictive_plot_kwargs = {
    "figure_size": (8, 6),
    "title": None,
    "title_font_size": 19,
    "show_legend": True,
    "legend_font_size": 12,
    "axis_label_font_size": 18,
    "tick_label_font_size": 14,
    "point_size": 32,
    "point_marker": "o",
    "prediction_marker": "o",
    "prediction_marker_size": 9,
    "context_obs_color": "forestgreen",
    "context_rem_color": "mediumseagreen",
}

print(f"Using experiment folder: {EXPERIMENT_DIR}")
print(f"Configured output root: {OUTPUT_ROOT}")


Using experiment folder: /home/cesarali/Pharma/pff/results/comet/uai/cb906bbb30f34522813c1357fff45371
Configured output root: /home/cesarali/Pharma/pff/reports/results_aistats/all_predictions


In [3]:
exp_config, model, datamodule, checkpoint_path = load_aicme_experiment(
    EXPERIMENT_DIR,
    checkpoint_type=CHECKPOINT_TYPE,
    map_location=MAP_LOCATION,
)

if FIX_PAST_TARGET_STEPS is not None:
    datamodule.fix_past_selection(FIX_PAST_TARGET_STEPS, who="target")
else:
    datamodule.release_past_selection(who="target")

device = torch.device(MAP_LOCATION)
print("Checkpoint:", checkpoint_path)
print("Model:", type(model).__name__)
print("Configured empirical datasets:", exp_config.mix_data.test_empirical_datasets)


Checkpoint: /home/cesarali/Pharma/pff/results/comet/uai/cb906bbb30f34522813c1357fff45371/scheduler_metric_checkpoints/empirical_summary/best-epoch_049-step_0003350-log_rmse=1.523099.ckpt
Model: AICMEPK
Configured empirical datasets: ['cesarali/lenuzza-2016', 'cesarali/Indometacin', 'cesarali/Theophylline']


In [4]:
selections = list(
    iter_first_batch_selections(
        datamodule,
        no_heldout=NO_HELDOUT,
        batch_index=BATCH_INDEX,
    )
)
if not selections:
    raise RuntimeError("No empirical selections were found for the requested batch index.")

selections_summary = pd.DataFrame.from_records(
    [
        {
            "dataset_key": selection["dataset_key"],
            "batch_index": selection["batch_index"],
            "selection_index": selection["selection_index"],
            "study_name": selection["study_name"],
            "drug_name": selection["drug_name"],
        }
        for selection in selections
    ]
)
print(f"Found {len(selections)} empirical Study/Drug selections in batch index {BATCH_INDEX}.")
selections_summary


Found 20 empirical Study/Drug selections in batch index 0.


,dataset_key,batch_index,selection_index,study_name,drug_name
0,cesarali/lenuzza-2016,0,0,Lenuzza2016,memantine
1,cesarali/lenuzza-2016,0,1,Lenuzza2016,omeprazole
2,cesarali/lenuzza-2016,0,2,Lenuzza2016,5-hydroxyomeprazole
3,cesarali/lenuzza-2016,0,3,Lenuzza2016,omeprazole sulfone
4,cesarali/lenuzza-2016,0,4,Lenuzza2016,repaglinide
5,cesarali/lenuzza-2016,0,5,Lenuzza2016,hydroxy repaglinide
6,cesarali/lenuzza-2016,0,6,Lenuzza2016,rosuvastatin
7,cesarali/lenuzza-2016,0,7,Lenuzza2016,tolbutamide
8,cesarali/lenuzza-2016,0,8,Lenuzza2016,4-hydroxytolbutamide
9,cesarali/lenuzza-2016,0,9,Lenuzza2016,dextromethorphan


In [5]:
prediction_records: list[dict[str, object]] = []
for index, selection in enumerate(selections, start=1):
    prediction_path = (
        selection_output_dir(selection)
        / f"{selection_file_stem(selection)}__prediction.png"
    )

    print(
        f"[prediction {index:03d}/{len(selections):03d}] "
        f"{selection['dataset_key']} | batch={selection['batch_index']} | "
        f"study={selection['study_name']} | drug={selection['drug_name']}"
    )

    prediction_saved = True
    prediction_status = "saved"
    resolved_study_name = str(selection["study_name"])
    resolved_drug_name = str(selection["drug_name"])
    try:
        resolved_metadata = save_prediction_plot(
            model=model,
            batch=selection["batch"],
            image_path=prediction_path,
            device=device,
            sample_size=SAMPLE_SIZE,
            log_scale=PLOT_LOG_SCALE,
            plot_kwargs=predictive_plot_kwargs,
        )
        resolved_study_name = resolved_metadata["study_name"]
        resolved_drug_name = resolved_metadata["drug_name"]
    except Exception as exc:
        prediction_saved = False
        prediction_status = f"failed: {exc}"
        prediction_path = None

    prediction_records.append(
        {
            "dataset_key": selection["dataset_key"],
            "batch_index": selection["batch_index"],
            "selection_index": selection["selection_index"],
            "study_name": resolved_study_name,
            "drug_name": resolved_drug_name,
            "prediction_saved": prediction_saved,
            "prediction_status": prediction_status,
            "prediction_path": str(prediction_path) if prediction_path is not None else None,
        }
    )

prediction_summary = pd.DataFrame.from_records(prediction_records)
prediction_summary = prediction_summary.sort_values(
    by=["dataset_key", "selection_index", "study_name", "drug_name"]
).reset_index(drop=True)
print(f"Saved {int(prediction_summary['prediction_saved'].sum())} prediction plot(s).")
prediction_summary


[prediction 001/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=memantine
[prediction 002/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=omeprazole
[prediction 003/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=5-hydroxyomeprazole
[prediction 004/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=omeprazole sulfone
[prediction 005/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=repaglinide
[prediction 006/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=hydroxy repaglinide
[prediction 007/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=rosuvastatin
[prediction 008/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=tolbutamide
[prediction 009/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=4-hydroxytolbutamide
[prediction 010/020] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=dextromethorphan
[prediction 011/020] cesarali/lenuzza-2016 | 

,dataset_key,batch_index,selection_index,study_name,drug_name,prediction_saved,prediction_status,prediction_path
0,cesarali/Indometacin,0,0,Indometacin,Indometacin,True,saved,/home/cesarali/Pharma/pff/reports/re...
1,cesarali/Theophylline,0,0,Theophylline,Theophylline,True,saved,/home/cesarali/Pharma/pff/reports/re...
2,cesarali/lenuzza-2016,0,0,Lenuzza2016,memantine,True,saved,/home/cesarali/Pharma/pff/reports/re...
3,cesarali/lenuzza-2016,0,1,Lenuzza2016,omeprazole,True,saved,/home/cesarali/Pharma/pff/reports/re...
4,cesarali/lenuzza-2016,0,2,Lenuzza2016,5-hydroxyomeprazole,True,saved,/home/cesarali/Pharma/pff/reports/re...
5,cesarali/lenuzza-2016,0,3,Lenuzza2016,omeprazole sulfone,True,saved,/home/cesarali/Pharma/pff/reports/re...
6,cesarali/lenuzza-2016,0,4,Lenuzza2016,repaglinide,True,saved,/home/cesarali/Pharma/pff/reports/re...
7,cesarali/lenuzza-2016,0,5,Lenuzza2016,hydroxy repaglinide,True,saved,/home/cesarali/Pharma/pff/reports/re...
8,cesarali/lenuzza-2016,0,6,Lenuzza2016,rosuvastatin,True,saved,/home/cesarali/Pharma/pff/reports/re...
9,cesarali/lenuzza-2016,0,7,Lenuzza2016,tolbutamide,True,saved,/home/cesarali/Pharma/pff/reports/re...
